# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassaanSaqib/FlyRankAI-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My confirmed lane is **Refresh / Content Opportunity Scoring**.

I use supervised binary classification because the retrospective evaluation target records whether a page was observed as declining (`trend_direction == "down"`). The model produces a probability rather than only a yes/no prediction, allowing pages to be ranked by observed decline risk for human review.

I train two methods:

- **Logistic Regression** as the simple and readable starting model.

- **Random Forest** to test whether non-linear relationships justify additional complexity.

The honest feature set uses only information available before the retrospective outcome is inspected. It excludes `trend_direction`, `trend_pct`, the derived evaluation label, and both identifiers. Missing values are handled with median imputation, while explicit availability indicators record whether important measurements were present.

My primary metric is **Precision@20** because the operational question is which 20 pages should be reviewed first. Precision@50, average precision, and ROC AUC are reported as supporting metrics. A more complex model is not selected unless its held-out performance earns that complexity

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

import subprocess

import numpy as np

import pandas as pd

import sklearn

from IPython.display import display, Markdown

from sklearn.model_selection import GroupShuffleSplit

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import average_precision_score, roc_auc_score

from sklearn.inspection import permutation_importance

# ------------------------------------------------------------

# 1A. Load the repository and starter dataset

# ------------------------------------------------------------

RANDOM_STATE = 42

REPO_URL = "https://github.com/HassaanSaqib/FlyRankAI-Internship.git"

REPO_DIR = Path("/content/flyrank_repo")

DATA_PATH = REPO_DIR / "data/raw/content_refresh_anonymized.csv"

if not REPO_DIR.exists():

    subprocess.run(

        ["git", "clone", "-q", REPO_URL, str(REPO_DIR)],

        check=True,

    )

else:

    # Safe in a temporary Colab runtime; keeps the cloned copy current.

    subprocess.run(

        ["git", "-C", str(REPO_DIR), "pull", "-q"],

        check=False,

    )

if not DATA_PATH.exists():

    raise FileNotFoundError(

        f"Starter dataset was not found at {DATA_PATH}"

    )

df = pd.read_csv(DATA_PATH)

required_columns = {

    "content_id",

    "client_id",

    "trend_direction",

    "impressions_90d",

    "clicks_90d",

    "sessions_90d",

    "ctr",

    "avg_position",

    "days_since_last_update",

    "content_age_days",

    "days_with_impressions",

    "engagement_rate",

    "scroll_rate",

    "word_count",

}

missing_columns = required_columns.difference(df.columns)

if missing_columns:

    raise KeyError(

        f"Required columns are missing: {sorted(missing_columns)}"

    )

numeric_source_columns = sorted(

    required_columns

    - {

        "content_id",

        "client_id",

        "trend_direction",

    }

)

for column in numeric_source_columns:

    df[column] = pd.to_numeric(

        df[column],

        errors="coerce",

    )

# ------------------------------------------------------------

# 1B. Retrospective evaluation target

# ------------------------------------------------------------

# This is used only as the observed evaluation outcome.

# trend_direction and trend_pct are NEVER model features.

df["is_declining_eval"] = (

    df["trend_direction"]

    .astype(str)

    .str.lower()

    .eq("down")

    .astype(int)

)

# ------------------------------------------------------------

# 1C. Honest feature engineering

# ------------------------------------------------------------

# Log transforms reduce the extreme skew in count variables.

df["log_impressions_90d"] = np.log1p(

    df["impressions_90d"].clip(lower=0)

)

df["log_clicks_90d"] = np.log1p(

    df["clicks_90d"].clip(lower=0)

)

df["log_sessions_90d"] = np.log1p(

    df["sessions_90d"].clip(lower=0)

)

# In this dataset, average position 0 means no position data.

df["avg_position_clean"] = df["avg_position"].where(

    df["avg_position"] > 0,

    np.nan,

)

# Explicit availability indicators prevent missing values from

# silently being treated as genuine zero measurements.

df["has_position"] = (

    df["avg_position_clean"].notna().astype(int)

)

df["has_word_count"] = (

    df["word_count"].notna().astype(int)

)

df["has_engagement_rate"] = (

    df["engagement_rate"].notna().astype(int)

)

df["has_scroll_rate"] = (

    df["scroll_rate"].notna().astype(int)

)

df["has_days_since_update"] = (

    df["days_since_last_update"].notna().astype(int)

)

feature_columns = [

    "log_impressions_90d",

    "log_clicks_90d",

    "log_sessions_90d",

    "ctr",

    "avg_position_clean",

    "days_since_last_update",

    "content_age_days",

    "days_with_impressions",

    "engagement_rate",

    "scroll_rate",

    "word_count",

    "has_position",

    "has_word_count",

    "has_engagement_rate",

    "has_scroll_rate",

    "has_days_since_update",

]

forbidden_features = {

    "content_id",

    "client_id",

    "trend_direction",

    "trend_pct",

    "is_declining_label",

    "is_declining_eval",

}

leaked_features = sorted(

    forbidden_features.intersection(feature_columns)

)

assert not leaked_features, (

    f"Leakage detected in feature list: {leaked_features}"

)

X = df[feature_columns].copy()

y = df["is_declining_eval"].astype(int)

# client_id is used only for grouped splitting.

groups = (

    df["client_id"]

    .fillna("unknown")

    .astype(str)

)

method_check = pd.DataFrame(

    {

        "item": [

            "Rows",

            "Clients",

            "Observed positive rate",

            "Model features",

            "Leaked features",

            "Random state",

            "scikit-learn version",

        ],

        "value": [

            f"{len(df):,}",

            f"{groups.nunique():,}",

            f"{y.mean():.1%}",

            len(feature_columns),

            len(leaked_features),

            RANDOM_STATE,

            sklearn.__version__,

        ],

    }

)

display(Markdown("### Method and data check"))

display(method_check)

print("Feature preparation complete.")

print("Leakage check: PASS")

print("Identifiers used as predictive features: none")

print("trend_direction used only as the evaluation outcome.")

### Method and data check

,item,value
0,Rows,"30,000"
1,Clients,32
2,Observed positive rate,54.2%
3,Model features,16
4,Leaked features,0
5,Random state,42
6,scikit-learn version,1.6.1


Feature preparation complete.
Leakage check: PASS
Identifiers used as predictive features: none
trend_direction used only as the evaluation outcome.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a **grouped client holdout split**. Approximately 80% of the pseudonymized clients are used for training and 20% are held out for testing. No client is allowed to appear in both sets.

This is more honest than a random row split because pages belonging to the same client may share traffic scale, content patterns, measurement practices, or other client-specific characteristics. A random row split could allow the model to learn those repeated patterns and then appear stronger when tested on other pages from the same clients.

The starter dataset is a trailing-90-day snapshot rather than a daily time panel, so a true chronological split is not available inside this file. Grouping by client is therefore the strongest available validation design for this Week-5 comparison.

The random state is fixed at 42 for reproducibility. The Week-4 baseline parameters are fitted using only the training clients, and the baseline and both models are then evaluated on exactly the same held-out rows.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ------------------------------------------------------------

# 2A. Create an honest grouped client split

# ------------------------------------------------------------

splitter = GroupShuffleSplit(

    n_splits=20,

    test_size=0.20,

    random_state=RANDOM_STATE,

)

valid_split = None

for candidate_train_idx, candidate_test_idx in splitter.split(

    X,

    y,

    groups,

):

    train_classes = y.iloc[candidate_train_idx].nunique()

    test_classes = y.iloc[candidate_test_idx].nunique()

    if train_classes == 2 and test_classes == 2:

        valid_split = (

            np.asarray(candidate_train_idx),

            np.asarray(candidate_test_idx),

        )

        break

if valid_split is None:

    raise ValueError(

        "A valid grouped split containing both target classes "

        "could not be created."

    )

train_idx, test_idx = valid_split

X_train = X.iloc[train_idx].copy()

X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()

y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])

test_clients = set(groups.iloc[test_idx])

client_overlap = train_clients.intersection(test_clients)

assert not client_overlap, (

    "Client leakage detected between training and testing sets."

)

split_table = pd.DataFrame(

    {

        "Split": ["Train", "Test"],

        "Rows": [

            len(train_idx),

            len(test_idx),

        ],

        "Clients": [

            len(train_clients),

            len(test_clients),

        ],

        "Observed positive rate": [

            y_train.mean(),

            y_test.mean(),

        ],

    }

)

display(

    split_table.style.format(

        {

            "Rows": "{:,.0f}",

            "Clients": "{:,.0f}",

            "Observed positive rate": "{:.1%}",

        }

    )

)

print("Split strategy: grouped client holdout")

print(f"Random state: {RANDOM_STATE}")

print(f"Client overlap: {len(client_overlap)}")

print("Grouped split leakage check: PASS")

,Split,Rows,Clients,Observed positive rate
0,Train,"23,837",25,55.0%
1,Test,"6,163",7,51.1%


Split strategy: grouped client holdout
Random state: 42
Client overlap: 0
Grouped split leakage check: PASS


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I reproduce the same Week-4 baseline logic rather than comparing the models with a different rule. The baseline requires at least 300 impressions and a valid search position. It identifies pages that are at least 180 days stale, have CTR at least 20% below the observed median for their position bucket, or meet both conditions. Visibility determines the ordering within those rule-based recommendations.

To prevent information from the held-out clients influencing the comparison, the position-bucket CTR benchmarks and score normalization values are calculated using only the training set. The fitted rule is then applied unchanged to the test set.

Logistic Regression and Random Forest use the same training rows and are evaluated on exactly the same test rows as the baseline. All three approaches produce continuous scores and are compared using the same ranking metrics. **Precision@20 is the primary selection metric**, with average precision used as the tie-breaker. Complexity alone is not treated as an improvement

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ------------------------------------------------------------

# 3A. Metric helpers

# ------------------------------------------------------------

def precision_at_k(y_true, scores, tie_break, k):

    """Observed positive rate among the k highest-ranked rows."""

    ranked = pd.DataFrame(

        {

            "target": np.asarray(y_true, dtype=int),

            "score": np.asarray(scores, dtype=float),

            "tie_break": np.asarray(tie_break, dtype=float),

        }

    )

    top_k = (

        ranked.sort_values(

            ["score", "tie_break"],

            ascending=[False, False],

            kind="mergesort",

        )

        .head(min(k, len(ranked)))

    )

    if top_k.empty:

        return 0.0

    return float(top_k["target"].mean())

def ranking_metrics(y_true, scores, tie_break):

    """Calculate the same ranking metrics for every method."""

    y_array = np.asarray(y_true, dtype=int)

    score_array = np.asarray(scores, dtype=float)

    return {

        "Test base rate": float(y_array.mean()),

        "Precision@20": precision_at_k(

            y_array,

            score_array,

            tie_break,

            20,

        ),

        "Precision@50": precision_at_k(

            y_array,

            score_array,

            tie_break,

            50,

        ),

        "Average precision": float(

            average_precision_score(

                y_array,

                score_array,

            )

        ),

        "ROC AUC": float(

            roc_auc_score(

                y_array,

                score_array,

            )

        ),

    }

# ------------------------------------------------------------

# 3B. Reproduce the Week-4 position buckets

# ------------------------------------------------------------

def position_bucket(frame):

    position = pd.to_numeric(

        frame["avg_position"],

        errors="coerce",

    )

    values = np.select(

        [

            position.isna() | position.le(0),

            position.le(3),

            position.le(10),

            position.le(20),

            position.le(50),

            position.gt(50),

        ],

        [

            "no_data",

            "top_3",

            "page_1",

            "striking",

            "page_3_5",

            "deep",

        ],

        default="no_data",

    )

    return pd.Series(

        values,

        index=frame.index,

    )

# ------------------------------------------------------------

# 3C. Fit and apply the same Week-4 rule baseline

# ------------------------------------------------------------

def score_week4_baseline(frame, parameters):

    scored = frame.copy()

    scored["position_bucket"] = position_bucket(scored)

    expected_ctr = pd.to_numeric(

        scored["position_bucket"].map(

            parameters["expected_ctr_map"]

        ),

        errors="coerce",

    ).fillna(

        parameters["global_expected_ctr"]

    )

    eligible = (

        scored["impressions_90d"].ge(300)

        & scored["avg_position"].gt(0)

    )

    stale = (

        scored["days_since_last_update"]

        .fillna(-1)

        .ge(180)

    )

    low_ctr = (

        eligible

        & expected_ctr.gt(0)

        & scored["ctr"].le(

            expected_ctr * 0.80

        )

    )

    signal_count = (

        stale.astype(int)

        + low_ctr.astype(int)

    )

    visibility_weight = (

        np.log1p(

            scored["impressions_90d"].clip(lower=0)

        )

        / np.log1p(

            parameters["impression_cap"]

        )

    ).clip(

        lower=0,

        upper=1,

    )

    raw_score = np.where(

        eligible,

        visibility_weight * signal_count,

        0.0,

    )

    normalized_score = (

        np.asarray(raw_score, dtype=float)

        / parameters.get(

            "max_raw_score",

            1.0,

        )

    )

    normalized_score = np.clip(

        normalized_score,

        0,

        1,

    )

    return pd.DataFrame(

        {

            "raw_score": raw_score,

            "baseline_score": normalized_score,

            "stale_flag": stale.to_numpy(),

            "low_ctr_flag": low_ctr.to_numpy(),

            "expected_ctr_for_position": (

                expected_ctr.to_numpy()

            ),

        },

        index=frame.index,

    )

def fit_week4_baseline(train_frame):

    train = train_frame.copy()

    train["position_bucket"] = position_bucket(train)

    position_audit = train[

        train["impressions_90d"].ge(300)

        & train["avg_position"].gt(0)

    ].copy()

    if position_audit.empty:

        raise ValueError(

            "No eligible training rows were available "

            "for the Week-4 baseline."

        )

    expected_ctr_map = (

        position_audit.groupby(

            "position_bucket",

            observed=False,

        )["ctr"]

        .median()

        .to_dict()

    )

    global_expected_ctr = float(

        position_audit["ctr"].median()

    )

    impression_cap = max(

        float(

            train["impressions_90d"].quantile(0.99)

        ),

        1.0,

    )

    parameters = {

        "expected_ctr_map": expected_ctr_map,

        "global_expected_ctr": global_expected_ctr,

        "impression_cap": impression_cap,

    }

    train_scores = score_week4_baseline(

        train,

        parameters,

    )

    parameters["max_raw_score"] = max(

        float(

            train_scores["raw_score"].max()

        ),

        1e-12,

    )

    return parameters

train_frame = df.iloc[train_idx].copy()

test_frame = df.iloc[test_idx].copy()

baseline_parameters = fit_week4_baseline(

    train_frame

)

baseline_test = score_week4_baseline(

    test_frame,

    baseline_parameters,

)

test_tie_break = (

    test_frame["impressions_90d"]

    .fillna(0)

    .to_numpy()

)

# ------------------------------------------------------------

# 3D. Train the candidate models

# ------------------------------------------------------------

model_pipelines = {

    "Logistic Regression": Pipeline(

        steps=[

            (

                "imputer",

                SimpleImputer(

                    strategy="median",

                ),

            ),

            (

                "scaler",

                StandardScaler(),

            ),

            (

                "model",

                LogisticRegression(

                    class_weight="balanced",

                    max_iter=2000,

                    random_state=RANDOM_STATE,

                ),

            ),

        ]

    ),

    "Random Forest": Pipeline(

        steps=[

            (

                "imputer",

                SimpleImputer(

                    strategy="median",

                ),

            ),

            (

                "model",

                RandomForestClassifier(

                    n_estimators=300,

                    max_depth=8,

                    min_samples_leaf=25,

                    class_weight="balanced_subsample",

                    random_state=RANDOM_STATE,

                    n_jobs=-1,

                ),

            ),

        ]

    ),

}

all_scores = {

    "Week-4 rule baseline": (

        baseline_test["baseline_score"].to_numpy()

    )

}

for model_name, pipeline in model_pipelines.items():

    pipeline.fit(

        X_train,

        y_train,

    )

    all_scores[model_name] = (

        pipeline.predict_proba(X_test)[:, 1]

    )

# ------------------------------------------------------------

# 3E. Same test set, same metrics, one comparison table

# ------------------------------------------------------------

metrics_by_method = {}

comparison_rows = []

for method_name, scores in all_scores.items():

    method_metrics = ranking_metrics(

        y_test,

        scores,

        test_tie_break,

    )

    metrics_by_method[method_name] = method_metrics

    comparison_rows.append(

        {

            "Method": method_name,

            **method_metrics,

        }

    )

comparison_table = pd.DataFrame(

    comparison_rows

)

comparison_table["Lift@20 vs base rate"] = (

    comparison_table["Precision@20"]

    / comparison_table["Test base rate"]

)

comparison_table = (

    comparison_table[

        [

            "Method",

            "Test base rate",

            "Precision@20",

            "Precision@50",

            "Lift@20 vs base rate",

            "Average precision",

            "ROC AUC",

        ]

    ]

    .sort_values(

        [

            "Precision@20",

            "Average precision",

        ],

        ascending=False,

    )

    .reset_index(drop=True)

)

display(Markdown("### Model-versus-baseline comparison"))

display(

    comparison_table.style.format(

        {

            "Test base rate": "{:.1%}",

            "Precision@20": "{:.1%}",

            "Precision@50": "{:.1%}",

            "Lift@20 vs base rate": "{:.2f}×",

            "Average precision": "{:.3f}",

            "ROC AUC": "{:.3f}",

        }

    )

)

# Select only between the learned models.

# The baseline remains the comparison reference.

model_only_names = list(model_pipelines.keys())

best_model_name = sorted(

    model_only_names,

    key=lambda name: (

        metrics_by_method[name]["Precision@20"],

        metrics_by_method[name]["Average precision"],

    ),

    reverse=True,

)[0]

best_model = model_pipelines[best_model_name]

best_scores = all_scores[best_model_name]

baseline_p20 = metrics_by_method[

    "Week-4 rule baseline"

]["Precision@20"]

best_model_p20 = metrics_by_method[

    best_model_name

]["Precision@20"]

if best_model_p20 > baseline_p20:

    comparison_verdict = (

        f"{best_model_name} beat the Week-4 rule baseline "

        f"on held-out Precision@20 "

        f"({best_model_p20:.1%} versus {baseline_p20:.1%})."

    )

elif best_model_p20 == baseline_p20:

    comparison_verdict = (

        f"{best_model_name} tied the Week-4 rule baseline "

        f"on held-out Precision@20 ({best_model_p20:.1%})."

    )

else:

    comparison_verdict = (

        f"{best_model_name} did not beat the Week-4 rule "

        f"baseline on held-out Precision@20 "

        f"({best_model_p20:.1%} versus {baseline_p20:.1%})."

    )

display(

    Markdown(

        f"""

**Selected learned method:** {best_model_name}

**Observed comparison:** {comparison_verdict}

This is a held-out association and ranking result. It does not prove

Google's algorithm or prove that refreshing a page would causally

change its future performance.

"""

    )

)

### Model-versus-baseline comparison

,Method,Test base rate,Precision@20,Precision@50,Lift@20 vs base rate,Average precision,ROC AUC
0,Logistic Regression,51.1%,80.0%,84.0%,1.57×,0.632,0.630
1,Random Forest,51.1%,45.0%,62.0%,0.88×,0.602,0.614
2,Week-4 rule baseline,51.1%,45.0%,48.0%,0.88×,0.528,0.545




**Selected learned method:** Logistic Regression

**Observed comparison:** Logistic Regression beat the Week-4 rule baseline on held-out Precision@20 (80.0% versus 45.0%).

This is a held-out association and ranking result. It does not prove

Google's algorithm or prove that refreshing a page would causally

change its future performance.



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I do not rely on the headline metric alone. I inspect the selected model in four ways:

1. I use permutation importance on the held-out test set to identify which inputs most affect average precision.

2. I calculate error rates across search-position buckets to find where the model is least reliable.

3. I review three concrete wrong cases using only public-safe, anonymized measurements.

4. I separate false positives from false negatives.

The 0.50 probability threshold is used only to make the error types easier to inspect. The primary assessment remains the ranked Precision@20 comparison. Feature importance is interpreted as model reliance, not as causal evidence that a feature controls search performance.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ------------------------------------------------------------

# 4A. Threshold-based diagnostic errors

# ------------------------------------------------------------

best_predictions = (

    best_scores >= 0.50

).astype(int)

error_frame = test_frame[

    [

        "impressions_90d",

        "ctr",

        "avg_position",

        "days_since_last_update",

        "content_age_days",

    ]

].copy()

error_frame["observed_label"] = (

    y_test.to_numpy()

)

error_frame["model_score"] = (

    best_scores

)

error_frame["predicted_label"] = (

    best_predictions

)

error_frame["position_bucket"] = (

    position_bucket(test_frame).to_numpy()

)

error_frame["error_type"] = np.select(

    [

        (

            error_frame["observed_label"].eq(0)

            & error_frame["predicted_label"].eq(1)

        ),

        (

            error_frame["observed_label"].eq(1)

            & error_frame["predicted_label"].eq(0)

        ),

    ],

    [

        "false_positive",

        "false_negative",

    ],

    default="correct",

)

error_frame["is_error"] = (

    error_frame["error_type"]

    .ne("correct")

    .astype(int)

)

error_counts = (

    error_frame["error_type"]

    .value_counts()

    .rename_axis("Error type")

    .reset_index(name="Rows")

)

display(Markdown("### Error counts"))

display(error_counts)

# ------------------------------------------------------------

# 4B. Error rates by search-position bucket

# ------------------------------------------------------------

position_error_table = (

    error_frame.groupby(

        "position_bucket",

        observed=False,

    )

    .agg(

        n=("is_error", "size"),

        observed_positive_rate=(

            "observed_label",

            "mean",

        ),

        mean_model_score=(

            "model_score",

            "mean",

        ),

        error_rate=(

            "is_error",

            "mean",

        ),

    )

    .reset_index()

    .sort_values(

        "error_rate",

        ascending=False,

    )

)

display(Markdown("### Errors by position bucket"))

display(

    position_error_table.style.format(

        {

            "n": "{:,.0f}",

            "observed_positive_rate": "{:.1%}",

            "mean_model_score": "{:.3f}",

            "error_rate": "{:.1%}",

        }

    )

)

# ------------------------------------------------------------

# 4C. Three concrete wrong cases

# ------------------------------------------------------------

false_positives = (

    error_frame[

        error_frame["error_type"].eq(

            "false_positive"

        )

    ]

    .sort_values(

        "model_score",

        ascending=False,

    )

    .head(2)

)

false_negatives = (

    error_frame[

        error_frame["error_type"].eq(

            "false_negative"

        )

    ]

    .sort_values(

        "model_score",

        ascending=True,

    )

    .head(2)

)

wrong_cases = pd.concat(

    [

        false_positives,

        false_negatives,

    ]

).head(3).copy()

wrong_cases.insert(

    0,

    "case",

    [

        f"case_{number}"

        for number in range(

            1,

            len(wrong_cases) + 1,

        )

    ],

)

wrong_case_columns = [

    "case",

    "error_type",

    "model_score",

    "impressions_90d",

    "ctr",

    "avg_position",

    "position_bucket",

    "days_since_last_update",

    "content_age_days",

    "observed_label",

    "predicted_label",

]

display(Markdown("### Three wrong cases"))

display(

    wrong_cases[

        wrong_case_columns

    ].style.format(

        {

            "model_score": "{:.3f}",

            "impressions_90d": "{:,.0f}",

            "ctr": "{:.2f}%",

            "avg_position": "{:.1f}",

            "days_since_last_update": "{:.0f}",

            "content_age_days": "{:.0f}",

        }

    )

)

# ------------------------------------------------------------

# 4D. Held-out permutation importance

# ------------------------------------------------------------

permutation_result = permutation_importance(

    best_model,

    X_test,

    y_test,

    scoring="average_precision",

    n_repeats=5,

    random_state=RANDOM_STATE,

    n_jobs=-1,

)

importance_table = (

    pd.DataFrame(

        {

            "feature": feature_columns,

            "importance_mean": (

                permutation_result.importances_mean

            ),

            "importance_std": (

                permutation_result.importances_std

            ),

        }

    )

    .sort_values(

        "importance_mean",

        ascending=False,

    )

    .reset_index(drop=True)

)

display(

    Markdown(

        f"### Held-out permutation importance — {best_model_name}"

    )

)

display(

    importance_table.head(10).style.format(

        {

            "importance_mean": "{:.4f}",

            "importance_std": "{:.4f}",

        }

    )

)

# ------------------------------------------------------------

# 4E. Generate a short observed interpretation

# ------------------------------------------------------------

top_three_features = (

    importance_table

    .head(3)["feature"]

    .tolist()

)

supported_groups = position_error_table[

    position_error_table["n"] >= 50

].copy()

if supported_groups.empty:

    hardest_group_text = (

        "No position bucket had enough held-out rows "

        "for a stable grouped error comparison."

    )

else:

    hardest_group = supported_groups.iloc[0]

    hardest_group_text = (

        f"The highest observed error rate among position "

        f"buckets with at least 50 rows was "

        f"`{hardest_group['position_bucket']}` "

        f"({hardest_group['error_rate']:.1%}, "

        f"n={int(hardest_group['n']):,})."

    )

false_positive_count = int(

    error_frame["error_type"]

    .eq("false_positive")

    .sum()

)

false_negative_count = int(

    error_frame["error_type"]

    .eq("false_negative")

    .sum()

)

top_feature_text = ", ".join(

    f"`{feature}`"

    for feature in top_three_features

)

display(

    Markdown(

        f"""

### Short error interpretation

The selected model relied most strongly on {top_feature_text}

according to held-out permutation importance. These variables are

plausibly related to observed visibility and page performance, but

their importance does not establish a causal ranking mechanism.

{hardest_group_text}

At the 0.50 diagnostic threshold, the model produced

**{false_positive_count:,} false positives** and

**{false_negative_count:,} false negatives**. The displayed wrong

cases show that pages with similar observed search measurements can

still have different retrospective outcomes. Therefore, the score

should support human review rather than automatically deciding that a

page must be refreshed.

"""

    )

)

print("Error analysis complete.")

print("Three wrong cases displayed.")

print("Held-out permutation importance calculated.")

### Error counts

,Error type,Rows
0,correct,3672
1,false_positive,1374
2,false_negative,1117


### Errors by position bucket

,position_bucket,n,observed_positive_rate,mean_model_score,error_rate
0,deep,279,35.5%,0.355,46.6%
4,striking,"1,461",50.1%,0.531,45.2%
3,page_3_5,"1,271",48.1%,0.478,45.2%
2,page_1,"2,820",55.7%,0.540,37.4%
5,top_3,270,50.4%,0.535,25.9%
1,no_data,62,1.6%,0.006,1.6%


### Three wrong cases

,case,error_type,model_score,impressions_90d,ctr,avg_position,position_bucket,days_since_last_update,content_age_days,observed_label,predicted_label
27993,case_1,false_positive,0.903,"1,266",0.00%,4.6,page_1,106,106,0,1
12869,case_2,false_positive,0.886,"15,101",0.00%,5.7,page_1,7,421,0,1
27271,case_3,false_negative,0.009,1,0.00%,0.0,no_data,92,238,1,0


### Held-out permutation importance — Logistic Regression

,feature,importance_mean,importance_std
0,log_clicks_90d,0.0897,0.0023
1,log_impressions_90d,0.0746,0.0061
2,avg_position_clean,0.0367,0.0014
3,content_age_days,0.0251,0.0029
4,log_sessions_90d,0.0178,0.0008
5,scroll_rate,0.0067,0.0030
6,has_word_count,0.0032,0.0016
7,has_position,0.0016,0.0017
8,has_scroll_rate,0.0008,0.0008
9,word_count,0.0003,0.0001




### Short error interpretation

The selected model relied most strongly on `log_clicks_90d`, `log_impressions_90d`, `avg_position_clean`

according to held-out permutation importance. These variables are

plausibly related to observed visibility and page performance, but

their importance does not establish a causal ranking mechanism.

The highest observed error rate among position buckets with at least 50 rows was `deep` (46.6%, n=279).

At the 0.50 diagnostic threshold, the model produced

**1,374 false positives** and

**1,117 false negatives**. The displayed wrong

cases show that pages with similar observed search measurements can

still have different retrospective outcomes. Therefore, the score

should support human review rather than automatically deciding that a

page must be refreshed.



Error analysis complete.
Three wrong cases displayed.
Held-out permutation importance calculated.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.